<a href="https://colab.research.google.com/github/mdanmek/nida-dads-notes/blob/main/dads5001-data-tools/project/eda/02_construction_data_preparation_2569.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DADS5001 Mini Project
## การเตรียมข้อมูลโครงการจ้างก่อสร้าง ปีงบประมาณ 2569

ข้อมูลชุดนี้เป็นข้อมูล e-GP ปีงบประมาณ 2569 **สะสมถึงวันที่ 30 กรกฎาคม 2569**
ไม่ใช่ข้อมูลเต็มปี Notebook อ่านไฟล์ต้นทาง 8 ไฟล์แบบแบ่งส่วน (chunk)
และสกัดเฉพาะประเภท `จ้างก่อสร้าง` สำหรับใช้ใน Notebook ถัดไป

ใช้ `รหัสโครงการ` เป็นหน่วยวิเคราะห์หลัก และเก็บทุกแถวไว้
สำหรับรายละเอียดสัญญา ผู้รับจ้าง และ Joint Venture (JV)

| ตัวแปรราคา | ความหมาย |
|---|---|
| วงเงินงบประมาณ | กรอบทรัพยากรของโครงการ |
| ราคากลาง | ราคาอ้างอิง |
| ราคาที่ตกลง | ราคาที่ตกลงรวมทุกสัญญาในโครงการ |

> ผลลัพธ์ใช้ค้นหารูปแบบเพื่อจัดลำดับการตรวจเอกสาร ไม่ใช่ข้อสรุปการทุจริต


In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.2f}'.format)

sns.set_theme(style='whitegrid')

base_dir = Path(
    '/content/drive/MyDrive/learning/dads/dads5001/'
    'project_1_dads5001/dataset/procurement/egp-contract/2569'
)

processed_dir = base_dir.parent / 'processed'
processed_dir.mkdir(parents=True, exist_ok=True)

csv_files = sorted(base_dir.glob('*.csv'))
construction_path = processed_dir / 'construction_contracts_2569.csv'

project_directory = (
    base_dir.parents[3]
    / 'dads5001-data-tools'
    / 'project'
    / 'eda'
)
figure_directory = (
    project_directory
    / 'figure'
)
figure_directory.mkdir(parents=True, exist_ok=True)

print(f'CSV files found: {len(csv_files)}')
print(f'Construction output: {construction_path}')
print(f'Figure directory: {figure_directory}')

## 1. สำรวจโครงสร้างข้อมูลเบื้องต้น

เริ่มจากอ่านข้อมูลตัวอย่างจากไฟล์แรก เพื่อทำความเข้าใจโครงสร้าง
ชื่อคอลัมน์ และลักษณะข้อมูล ก่อนประมวลผลไฟล์ทั้งหมด

In [ ]:
sample_data = pd.read_csv(
    csv_files[0],
    nrows=5
)

print(f'Sample file: {csv_files[0].name}')
print(f'Number of columns: {sample_data.shape[1]}')

display(sample_data)

In [ ]:
for position, column in enumerate(
    sample_data.columns,
    start=1
):
    print(f'{position:02d}. {column}')

## 2. ภาพรวมประเภทโครงการและการสกัดข้อมูล

ข้อมูลทั้งหมดจะถูกอ่านแบบแบ่งส่วน เพื่อทำงานสองอย่างในรอบเดียว:

1. นับจำนวนรายการตามประเภทโครงการ เพื่อดูภาพรวมของข้อมูล
2. สกัดรายการ `จ้างก่อสร้าง` ทุกคอลัมน์ออกมาเป็นไฟล์ใหม่

ในขั้นนี้ใช้คำว่า “จำนวนรายการ” เนื่องจากหนึ่งโครงการอาจมี
หลายสัญญาและปรากฏได้มากกว่าหนึ่งแถว

In [ ]:
project_type_column = 'ชื่อประเภทโครงการ'
chunk_size = 100_000

type_counts_list = []
processing_results = []

first_write = True

for file_number, file_path in enumerate(
    csv_files,
    start=1
):
    print(
        f'Processing {file_number}/{len(csv_files)}: '
        f'{file_path.name}'
    )

    total_rows = 0
    construction_rows = 0

    reader = pd.read_csv(
        file_path,
        chunksize=chunk_size,
        low_memory=False
    )

    for chunk in reader:
        chunk.columns = chunk.columns.str.strip()
        total_rows += len(chunk)

        project_type = (
            chunk[project_type_column]
            .astype('string')
            .str.strip()
            .fillna('ไม่ระบุ')
        )

        type_counts_list.append(
            project_type.value_counts()
        )

        construction_chunk = chunk.loc[
            project_type.eq('จ้างก่อสร้าง')
        ].copy()

        construction_rows += len(
            construction_chunk
        )

        if not construction_chunk.empty:
            construction_chunk['source_file'] = (
                file_path.name
            )

            construction_chunk.to_csv(
                construction_path,
                mode='w' if first_write else 'a',
                header=first_write,
                index=False,
                encoding='utf-8-sig'
            )

            first_write = False

    processing_results.append({
        'file_name': file_path.name,
        'total_rows': total_rows,
        'construction_rows': construction_rows
    })

    print(
        f'  Total rows: {total_rows:,} | '
        f'Construction rows: {construction_rows:,}'
    )

processing_summary = pd.DataFrame(
    processing_results
)

project_type_counts = (
    pd.concat(
        type_counts_list,
        axis=1
    )
    .fillna(0)
    .sum(axis=1)
    .astype('int64')
    .sort_values(ascending=False)
    .rename('record_count')
    .reset_index()
    .rename(
        columns={
            'index': project_type_column
        }
    )
)

total_records = (
    processing_summary['total_rows'].sum()
)

total_construction_records = (
    processing_summary[
        'construction_rows'
    ].sum()
)

construction_pct = (
    total_construction_records
    / total_records
    * 100
)

display(processing_summary)

print(f'Total records: {total_records:,}')
print(
    f'Construction records: '
    f'{total_construction_records:,}'
)
print(
    f'Construction share: '
    f'{construction_pct:.2f}%'
)
print(f'Output file: {construction_path}')

In [ ]:
# ตรวจยืนยันผลการประมวลผลกับข้อมูลชุดที่ใช้ในโครงการ
expected_file_count = 8
expected_total_records = 3_964_924
expected_construction_records = 180_079

assert len(csv_files) == expected_file_count, (
    f'Expected {expected_file_count} source files, '
    f'but found {len(csv_files)}'
)
assert total_records == expected_total_records, (
    f'Expected {expected_total_records:,} source rows, '
    f'but found {total_records:,}'
)
assert total_construction_records == expected_construction_records, (
    f'Expected {expected_construction_records:,} construction rows, '
    f'but found {total_construction_records:,}'
)
assert construction_path.exists(), (
    f'Output file was not created: {construction_path}'
)

print('Validation passed:')
print(f'- Source files: {len(csv_files)}')
print(f'- Source rows: {total_records:,}')
print(f'- Construction rows: {total_construction_records:,}')
print(f'- Output file exists: {construction_path.exists()}')

### ผลการประมวลผล

ข้อมูลต้นทางปีงบประมาณ 2569 สะสมถึงวันที่ 30 กรกฎาคม 2569 ประกอบด้วย 8 ไฟล์ รวมทั้งหมด
3,964,924 รายการ

เมื่อกรองด้วย `ชื่อประเภทโครงการ = 'จ้างก่อสร้าง'`
พบข้อมูลจำนวน 180,079 รายการ คิดเป็น 4.54% ของข้อมูลทั้งหมด
และได้บันทึกข้อมูลกลุ่มดังกล่าวเป็นไฟล์
`construction_contracts_2569.csv` สำหรับใช้วิเคราะห์ต่อ

จำนวนดังกล่าวเป็นจำนวนแถวหรือรายการในข้อมูล ไม่ใช่จำนวนโครงการ
ที่ไม่ซ้ำ เนื่องจากหนึ่งโครงการอาจมีมากกว่าหนึ่งสัญญา
หรือมากกว่าหนึ่งผู้ชนะการเสนอราคา

In [ ]:
project_type_counts['record_pct'] = (
    project_type_counts['record_count']
    .div(
        project_type_counts[
            'record_count'
        ].sum()
    )
    .mul(100)
)

display(project_type_counts)

In [ ]:
major_types = (
    project_type_counts
    .head(4)
    .sort_values(
        'record_pct',
        ascending=True
    )
    .copy()
)

label_map = {
    'ซื้อ': 'Purchase',
    'จ้างทำของ/จ้างเหมาบริการ': 'Service',
    'จ้างก่อสร้าง': 'Construction',
    'เช่า': 'Rental'
}

major_types['project_type_en'] = (
    major_types['ชื่อประเภทโครงการ']
    .map(label_map)
)

colors = [
    '#E67E22'
    if project_type == 'จ้างก่อสร้าง'
    else '#8FA3B8'
    for project_type
    in major_types['ชื่อประเภทโครงการ']
]

fig, ax = plt.subplots(
    figsize=(10, 5)
)

bars = ax.barh(
    major_types['project_type_en'],
    major_types['record_pct'],
    color=colors
)

ax.bar_label(
    bars,
    labels=[
        f'{value:.2f}%'
        for value
        in major_types['record_pct']
    ],
    padding=4
)

ax.set_title(
    'Share of Procurement Records by Major Project Type'
)
ax.set_xlabel('Share of records (%)')
ax.set_ylabel('Project type')
ax.set_xlim(
    0,
    major_types['record_pct'].max() * 1.12
)

ax.spines[
    ['top', 'right', 'left']
].set_visible(False)

plt.tight_layout()

figure_path = figure_directory / 'fig02_01_share_of_procurement_record.png'
fig.savefig(
    figure_path,
    dpi=150,
    bbox_inches='tight'
)

plt.show()
print(f'Figure saved: {figure_path}')

### ภาพรวมประเภทโครงการ

`จ้างก่อสร้าง` มี 180,079 รายการ หรือ 4.54% ของจำนวนแถวทั้งหมด
ตัวเลขนี้ไม่ใช่สัดส่วนมูลค่า แต่มีข้อมูลเพียงพอสำหรับวิเคราะห์
ขนาดโครงการ ราคา วิธีจัดซื้อ หน่วยงาน พื้นที่ และผู้รับจ้าง

ข้อมูลถูกบันทึกเป็น `construction_contracts_2569.csv`
เพื่อใช้ใน Notebook ถัดไปโดยไม่ต้องอ่านไฟล์ต้นทางประมาณ 4 GB ซ้ำ


## 3. ตรวจสอบชุดข้อมูลจ้างก่อสร้าง

ก่อนจบขั้นตอนเตรียมข้อมูล จะโหลดไฟล์จ้างก่อสร้างที่สร้างขึ้น
เพื่อตรวจสอบจำนวนแถว จำนวนคอลัมน์ และตัวอย่างข้อมูล
ให้แน่ใจว่าไฟล์พร้อมสำหรับใช้ใน Notebook EDA

In [ ]:
construction_data = pd.read_csv(
    construction_path,
    low_memory=False
)

print(
    f'Shape: '
    f'{construction_data.shape}'
)

display(
    construction_data.head()
)

In [ ]:
validation_summary = pd.Series({
    'Rows': len(construction_data),
    'Columns': construction_data.shape[1],
    'Unique project IDs': (
        construction_data[
            'รหัสโครงการ'
        ].nunique()
    ),
    'Unique contract number labels (not contract count)': (
        construction_data[
            'เลขที่สัญญา'
        ].nunique()
    ),
    'Missing project IDs': (
        construction_data[
            'รหัสโครงการ'
        ].isna().sum()
    ),
    'Missing contract numbers': (
        construction_data[
            'เลขที่สัญญา'
        ].isna().sum()
    ),
    'Exact duplicate rows': (
        construction_data
        .drop(columns='source_file')
        .duplicated()
        .sum()
    )
})

display(
    validation_summary.to_frame(
        name='value'
    )
)

display(
    construction_data[
        'ชื่อประเภทโครงการ'
    ].value_counts(
        dropna=False
    )
)

In [ ]:
project_row_counts = (
    construction_data[
        'รหัสโครงการ'
    ]
    .value_counts()
)

print(
    'Projects with more than one row:',
    (project_row_counts > 1).sum()
)

print(
    'Maximum rows per project:',
    project_row_counts.max()
)

print(
    '\nMost frequent contract number labels '
    '(not a count of unique contracts):'
)

display(
    construction_data[
        'เลขที่สัญญา'
    ]
    .value_counts(
        dropna=False
    )
    .head(10)
)

repeated_project_ids = (
    project_row_counts.loc[
        project_row_counts > 1
    ]
    .head(3)
    .index
)

repeated_project_sample = (
    construction_data.loc[
        construction_data[
            'รหัสโครงการ'
        ].isin(
            repeated_project_ids
        ),
        [
            'รหัสโครงการ',
            'ชื่อโครงการจัดซื้อจัดจ้าง',
            'ชื่อผู้ชนะการเสนอราคา',
            'เลขที่สัญญา',
            'วงเงินงบประมาณในสัญญา (บาท)'
        ]
    ]
    .sort_values(
        [
            'รหัสโครงการ',
            'เลขที่สัญญา'
        ]
    )
    .head(20)
)

display(repeated_project_sample)

### ผลลัพธ์และการส่งต่อ

ข้อมูลจ้างก่อสร้างมี 180,079 แถว 29 คอลัมน์ และ 178,978 โครงการไม่ซ้ำ
พบ 524 โครงการที่มีมากกว่าหนึ่งแถว โดยสูงสุด 15 แถวต่อโครงการ
จึงไม่ลบแถวซ้ำรหัสโครงการในขั้นเตรียมข้อมูล

Notebook 03 จะสำรวจขนาดโครงการ วิธีจัดซื้อ จุดกระจุกใกล้ 500,000 บาท
และความสัมพันธ์ของงบประมาณ ราคากลาง และราคาที่ตกลง
